In [ ]:
# Running inference for AlphaFold3 with pooled input, these are used on euler in a snakemake pipeline with cluster-specific optimisations:
#   https://github.com/jurgjn/batch-infer/blob/develop/examples/alphafold3_datafill/run_alphafold.ipynb
#
# Repository contains three pool input JSONs under alphafold3_jsons/
# Pools are identified by the SHA hash of the uniprot identifers
!ls -1 alphafold3_jsons/

a6455ba32ceb073e08e70339ba26f734f56b7b45.json
b42b6f73d193f543feece553ff1124226d5d105a.json
f4763cfa77756b51c269f4ff8d4f18e1b0a2cef2.json


In [3]:
# If af3io not available, install with `pip install af3io`
# Input JSONs only have the sequences, data pipeline output is added from the monomer MSAs using af3io
# This notebook runs inference for one pool - b42b6f73d193f543feece553ff1124226d5d105a
!af3io input-show alphafold3_jsons/b42b6f73d193f543feece553ff1124226d5d105a.json

{
  "dialect": "alphafold3",
  "version": 2,
  "name": "b42b6f73d193f543feece553ff1124226d5d105a",
  "sequences": [
    {
      "protein": {
        "id": "A",
        "sequence": "MGTSSEVSLAHHRDIFHYYVSLKTFFEVTGENRDRSNSTRAQKARAKLLKLSSSQFYELSTDVSDELQRRIGEDANQPDYLLPKANFHMKRNQARQKLANLSQTRFNDLLDDILFEIKRRGFDKDLDAPRPPLPQPMKQEVSKDSDDTARTSTNSSSVTQVAPNVSVQPSLVIPKMASIDWSSEEEEEEQVKEKPNEPEGKQTSMDEKKEAKPALNPIVTDSDLPDSQVLARDITSMARTPTTTHKNYWDVNDSPIIKVDKDIDNEKGPEQLKSPEVQRAENNNPNSEMEDKVKELTDLNSDLHLQIEDLNAKLASLTSEKEKEKKEEKEEKEKEKNLKINYTIDESFQKELLSLNSQIGELSIENENLKQKISEFELHQKKNDNHNDLKITDGFISKYSSADGLIPAQYILNANNLIIQFTTRLSAVPIGDSTAISHQIGEELFQILSQLSNLISQLLLSADLLQYKDQVILLKASLSHAITSIRYFSVYGPVLIPKITVQAAVSEVCFAMCNLIDSAKIKSDSNGESTTSNEGNRQVLEYSSPTATTPMTPTFPSTSGINMKKGFINPRKPASFLNDVEEEESPVKPLKITQKAINSPIIRPSSSNGVPTTSRKPSGTGLFSLMIDSSIAKNSSHKEDNDKYVSPIKAVTSASNSASSNISEIPKLTLPPQAKIGTVIPPSENQVPNIKIENTEEDNKRSDITNEISVKPTSSIADKLKQFEQSSEKKSSPKENPIAKEEMDSKPKLSNKFITSMNDVSTDDSSSDGNENDDADDDDDFTYMALKQTMKREGSKIEKNNDSKLPANIVELDLHESPE

In [ ]:
# Download and extract monomer data pipeline output from zenodo (run once, not for every pool)
!curl -s https://zenodo.org/records/18925033/files/pooled-ppi-yeast-msas.tar?download=1 | tar xvf -
# Should end up with 6,039 files under alphafold3_monomer_msas/
!mv alphafold3_msas alphafold3_monomer_msas
!ls -1 alphafold3_monomer_msas/*.json.gz | wc -l

alphafold3_msas/a0a023pyf4_data.json.gz
alphafold3_msas/a0a023pzb3_data.json.gz
alphafold3_msas/a0a023pze8_data.json.gz
alphafold3_msas/a0a023pzf2_data.json.gz
alphafold3_msas/a0a023pzg4_data.json.gz
alphafold3_msas/a0a0b7p221_data.json.gz
alphafold3_msas/a0a0b7p3v8_data.json.gz
alphafold3_msas/a0a7m4b2u1_data.json.gz
alphafold3_msas/a2p2r3_data.json.gz
alphafold3_msas/a5z2x5_data.json.gz
alphafold3_msas/d6vpm8_data.json.gz
alphafold3_msas/d6vtk4_data.json.gz
alphafold3_msas/d6w196_data.json.gz
alphafold3_msas/i2hb52_data.json.gz
alphafold3_msas/i2hb70_data.json.gz
alphafold3_msas/o13297_data.json.gz
alphafold3_msas/o13329_data.json.gz
alphafold3_msas/o13511_data.json.gz
alphafold3_msas/o13512_data.json.gz
alphafold3_msas/o13516_data.json.gz
alphafold3_msas/o13525_data.json.gz
alphafold3_msas/o13527_data.json.gz
alphafold3_msas/o13529_data.json.gz
alphafold3_msas/o13535_data.json.gz
alphafold3_msas/o13536_data.json.gz
alphafold3_msas/o13539_data.json.gz
alphafold3_msas/o13544_data.json

In [5]:
# Create an "index" with the sequences from the monomer MSAs (run once, not for every pool)
!af3io data-fill --data_dir alphafold3_monomer_msas --write-index

Create data index from: alphafold3_monomer_msas
Reading files:  [####################################]  100%                               
Read 5973 protein, 0 dna, 0 rna sequence(s)
Writing index to: alphafold3_monomer_msas/.af3io_data_index.json
Data index has 5973 protein, 0 dna, 0 rna sequence(s)


In [6]:
# The index is just a plain-text .json with the protein sequence, and the absolute path of the data pipeline output
!head -n 5 alphafold3_monomer_msas/.af3io_data_index.json

{
  "protein": {
    "MAILLPLKSILPWCCITFSFLLSSSGSISHSTASSSITLTKSSKPTNVPSNSRFDCSTINTFWLIVLSMTSKGKISGRLILRASVYACECTCIRYACCETIYPPRKPFSLSLYFFYFNKKASILFCYPDAKTKPEHPGNKRAGSG": "/cluster/project/beltrao/jjaenes/25.12.03_pooled-ppi/results-yeast/examples/run_alphafold3_pool_inference/alphafold3_monomer_msas/a0a023pyf4_data.json.gz",
    "MYYFSRVAARTFCCCIFFCLATAYSRPDRNPRKIEKKDKKFFGASKNTNPANAMGNLFKAPTIEYVVEEVTRTHQPEQYDIPTDMSPLMTIAASESADKFTDKFFVDQSSIMKEKTSSKGNARTLL": "/cluster/project/beltrao/jjaenes/25.12.03_pooled-ppi/results-yeast/examples/run_alphafold3_pool_inference/alphafold3_monomer_msas/a0a023pzb3_data.json.gz",
    "MSETCSSSLALLHKILHIHSHTPSVYYNICISVRILTSERLQCFFFSFFPDPNITGSGLKVPGFLFFHTFFFSKSCCQALIDSFSSDYYQFKMLEKNRKAEKINKRTIFICSFTFEYKIKSCFSCFHLSTHTN": "/cluster/project/beltrao/jjaenes/25.12.03_pooled-ppi/results-yeast/examples/run_alphafold3_pool_inference/alphafold3_monomer_msas/a0a023pze8_data.json.gz",


In [7]:
# Just before running inference, "fill in" the data pipeline strings from the monomer MSAs
# On euler, the "filled in" JSON is stored on local scratch & discarded after inference
!af3io data-fill \
    --json_path alphafold3_jsons/b42b6f73d193f543feece553ff1124226d5d105a.json \
    --data_dir alphafold3_monomer_msas \
    --output_dir alphafold3_msas

Load data index from: alphafold3_monomer_msas/.af3io_data_index.json
Read 5973 protein, 0 dna, 0 rna sequence(s)
Data index has 5973 protein, 0 dna, 0 rna sequence(s)
Read:	alphafold3_jsons/b42b6f73d193f543feece553ff1124226d5d105a.json
	fill id=A from id=A in /cluster/project/beltrao/jjaenes/25.12.03_pooled-ppi/results-yeast/examples/run_alphafold3_pool_inference/alphafold3_monomer_msas/p23201_data.json.gz
	fill id=B from id=A in /cluster/project/beltrao/jjaenes/25.12.03_pooled-ppi/results-yeast/examples/run_alphafold3_pool_inference/alphafold3_monomer_msas/p38737_data.json.gz
	fill id=C from id=A in /cluster/project/beltrao/jjaenes/25.12.03_pooled-ppi/results-yeast/examples/run_alphafold3_pool_inference/alphafold3_monomer_msas/p47033_data.json.gz
	fill id=D from id=A in /cluster/project/beltrao/jjaenes/25.12.03_pooled-ppi/results-yeast/examples/run_alphafold3_pool_inference/alphafold3_monomer_msas/q04213_data.json.gz
	fill id=E from id=A in /cluster/project/beltrao/jjaenes/25.12.03_po

In [8]:
# There's now an uncompressed JSON with the megabytes-long data pipeline strings at:
#   alphafold3_msas/b42b6f73d193f543feece553ff1124226d5d105a_data.json
!af3io input-show alphafold3_msas/b42b6f73d193f543feece553ff1124226d5d105a_data.json

{
  "dialect": "alphafold3",
  "version": 2,
  "name": "b42b6f73d193f543feece553ff1124226d5d105a",
  "sequences": [
    {
      "protein": {
        "id": "A",
        "sequence": "MGTSSEVSLAHHRDIFHYYVSLKTFFEVTGENRDRSNSTRAQKARAKLLKLSSSQFYELSTDVSDELQRRIGEDANQPDYLLPKANFHMKRNQARQKLANLSQTRFNDLLDDILFEIKRRGFDKDLDAPRPPLPQPMKQEVSKDSDDTARTSTNSSSVTQVAPNVSVQPSLVIPKMASIDWSSEEEEEEQVKEKPNEPEGKQTSMDEKKEAKPALNPIVTDSDLPDSQVLARDITSMARTPTTTHKNYWDVNDSPIIKVDKDIDNEKGPEQLKSPEVQRAENNNPNSEMEDKVKELTDLNSDLHLQIEDLNAKLASLTSEKEKEKKEEKEEKEKEKNLKINYTIDESFQKELLSLNSQIGELSIENENLKQKISEFELHQKKNDNHNDLKITDGFISKYSSADGLIPAQYILNANNLIIQFTTRLSAVPIGDSTAISHQIGEELFQILSQLSNLISQLLLSADLLQYKDQVILLKASLSHAITSIRYFSVYGPVLIPKITVQAAVSEVCFAMCNLIDSAKIKSDSNGESTTSNEGNRQVLEYSSPTATTPMTPTFPSTSGINMKKGFINPRKPASFLNDVEEEESPVKPLKITQKAINSPIIRPSSSNGVPTTSRKPSGTGLFSLMIDSSIAKNSSHKEDNDKYVSPIKAVTSASNSASSNISEIPKLTLPPQAKIGTVIPPSENQVPNIKIENTEEDNKRSDITNEISVKPTSSIADKLKQFEQSSEKKSSPKENPIAKEEMDSKPKLSNKFITSMNDVSTDDSSSDGNENDDADDDDDFTYMALKQTMKREGSKIEKNNDSKLPANIVELDLHESPE

In [9]:
# Run inference with a container (singularity, podman, ...):
# - There's no need to specify the database paths as we're only running the inference step
# - Use three recycles to reduce runtime
!mkdir -p alphafold3_predictions
!singularity exec --nv --writable-tmpfs \
    --bind alphafold3_msas:/root/af_input \
    --bind alphafold3_predictions:/root/af_output \
    --bind $HOME/.alphafold3_model_dir:/root/models \
    $SHARED/alphafold3/images/alphafold3_b78e215.sif \
        sh -c 'python /app/alphafold/run_alphafold.py --norun_data_pipeline --num_recycles=3 --input_dir=/root/af_input --output_dir=/root/af_output --model_dir=/root/models'

/alphafold3_venv/lib/python3.12/site-packages/keras/src/export/tf2onnx_lib.py:8: FutureWarning: In the future `np.object` will be defined as the corresponding NumPy scalar.
  if not hasattr(np, "object"):
W0000 00:00:1773155024.220629 3873795 gpu_device.cc:2431] TensorFlow was not built with CUDA kernel binaries compatible with compute capability 12.0. CUDA kernels will be jit-compiled from PTX, which could take 30 minutes or longer.
INFO:2026-03-10 16:03:45,572:jax._src.xla_bridge:812: Unable to initialize backend 'tpu': INTERNAL: Failed to open libtpu.so: libtpu.so: cannot open shared object file: No such file or directory
I0310 16:03:45.572376 22750781444864 xla_bridge.py:812] Unable to initialize backend 'tpu': INTERNAL: Failed to open libtpu.so: libtpu.so: cannot open shared object file: No such file or directory
W0310 16:03:45.845690 3873866 cuda_executor.cc:1802] GPU interconnect information not available: INTERNAL: NVML doesn't support extracting fabric info or NVLink is not us

In [ ]:
# Create a .zip file with models and confidences, exclude data pipeline output
!cd alphafold3_predictions && zip -r b42b6f73d193f543feece553ff1124226d5d105a.zip b42b6f73d193f543feece553ff1124226d5d105a/ \
    -x b42b6f73d193f543feece553ff1124226d5d105a/b42b6f73d193f543feece553ff1124226d5d105a_data.json

  adding: b42b6f73d193f543feece553ff1124226d5d105a/ (stored 0%)
  adding: b42b6f73d193f543feece553ff1124226d5d105a/seed-4_sample-0/ (stored 0%)
  adding: b42b6f73d193f543feece553ff1124226d5d105a/seed-4_sample-0/b42b6f73d193f543feece553ff1124226d5d105a_seed-4_sample-0_model.cif (deflated 75%)
  adding: b42b6f73d193f543feece553ff1124226d5d105a/seed-4_sample-0/b42b6f73d193f543feece553ff1124226d5d105a_seed-4_sample-0_summary_confidences.json (deflated 65%)
  adding: b42b6f73d193f543feece553ff1124226d5d105a/seed-4_sample-0/b42b6f73d193f543feece553ff1124226d5d105a_seed-4_sample-0_confidences.json (deflated 94%)
  adding: b42b6f73d193f543feece553ff1124226d5d105a/seed-4_sample-1/ (stored 0%)
  adding: b42b6f73d193f543feece553ff1124226d5d105a/seed-4_sample-1/b42b6f73d193f543feece553ff1124226d5d105a_seed-4_sample-1_model.cif (deflated 75%)
  adding: b42b6f73d193f543feece553ff1124226d5d105a/seed-4_sample-1/b42b6f73d193f543feece553ff1124226d5d105a_seed-4_sample-1_summary_confidences.json (deflated

In [11]:
# Double-check compressed output
!zip --test alphafold3_predictions/b42b6f73d193f543feece553ff1124226d5d105a.zip

test of alphafold3_predictions/b42b6f73d193f543feece553ff1124226d5d105a.zip OK


In [12]:
# Discard uncompressed output
!rm -rf alphafold3_predictions/b42b6f73d193f543feece553ff1124226d5d105a

In [ ]:
# Output one .zip file per pool, size varies, up to 200M
!ls -lh alphafold3_predictions/

total 78M
-rw-r--r-- 1 jjaenes jjaenes-group 78M Mar 10 16:19 b42b6f73d193f543feece553ff1124226d5d105a.zip
